# 🏠 NhaDat Chatbot - PDF Trainer
**Mục đích:** Đọc PDF từ Google Drive → dùng Gemini Vision trích xuất nội dung → push lên GitHub để bot học.

### Hướng dẫn:
1. Vào **Secrets (🔑 biểu tượng chìa khóa bên trái)** → thêm `GEMINI_API_KEY` và `GITHUB_TOKEN`
2. Sửa `DRIVE_FOLDER` ở Cell 2 cho đúng đường dẫn folder PDF trong Drive
3. **Runtime → Run all**

In [ ]:
# ── CELL 1: Cài thư viện ──────────────────────────────────────
!pip install -q pymupdf pdfplumber Pillow requests
print('✅ Cài xong thư viện')

In [ ]:
# ── CELL 2: CẤU HÌNH - SỬA Ở ĐÂY ────────────────────────────
import os, json, base64, time, requests
from pathlib import Path
from google.colab import drive, userdata

# ⬇️ SỬA ĐƯỜNG DẪN FOLDER PDF TRONG GOOGLE DRIVE
DRIVE_FOLDER = 'MyDrive/NhaDat/PDFs'
# ⬆️ Ví dụ: 'MyDrive/ChatBotData' hoặc 'Shared drives/Team/PDFs'

GITHUB_OWNER  = 'quang507'
GITHUB_REPO   = 'NhaDat-chatbot'
GITHUB_BRANCH = 'main'
OUTPUT_DIR    = '/content/output_md'

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GITHUB_TOKEN   = userdata.get('GITHUB_TOKEN')

os.makedirs(OUTPUT_DIR, exist_ok=True)
drive.mount('/content/drive', force_remount=False)

FULL_PATH = f'/content/drive/{DRIVE_FOLDER}'
print(f'📁 Folder: {FULL_PATH}')
print(f'🔑 Gemini: {"✅" if GEMINI_API_KEY else "❌ CHƯA SET TRONG SECRETS"}')
print(f'🔑 GitHub: {"✅" if GITHUB_TOKEN else "❌ CHƯA SET TRONG SECRETS"}')
print(f'📂 Folder tồn tại: {os.path.exists(FULL_PATH)}')

In [ ]:
# ── CELL 3: Hàm xử lý PDF ────────────────────────────────────
import fitz, pdfplumber

GEMINI_URL = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}'

PROMPT = '''Bạn là chuyên gia trích xuất dữ liệu bất động sản.
Đọc hình ảnh trang PDF và chuyển TOÀN BỘ nội dung thành Markdown có cấu trúc.
- Giữ NGUYÊN mọi số liệu: diện tích, giá, mã căn/lô, kích thước
- Bảng → Markdown table | Tiêu đề → ## | Danh sách → bullet
- Sơ đồ/mặt bằng: liệt kê từng lô với số hiệu và diện tích
- KHÔNG thêm nhận xét, chỉ chuyển dữ liệu
Trả về Markdown thuần, không dùng code block.'''

def page_to_image(page, dpi=200):
    mat = fitz.Matrix(dpi/72, dpi/72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return pix.tobytes('png')

def gemini_ocr(img_bytes, retry=3):
    import base64
    b64 = base64.b64encode(img_bytes).decode()
    payload = {'contents': [{'parts': [
        {'text': PROMPT},
        {'inline_data': {'mime_type': 'image/png', 'data': b64}}
    ]}], 'generationConfig': {'temperature': 0.1, 'maxOutputTokens': 8192}}
    for i in range(retry):
        r = requests.post(GEMINI_URL, json=payload, timeout=90)
        if r.status_code == 429:
            w = (i+1)*20; print(f'⏳ Rate limit, chờ {w}s...'); time.sleep(w); continue
        r.raise_for_status()
        return r.json()['candidates'][0]['content']['parts'][0]['text']
    return ''

def extract_text_page(pdf_path, page_num):
    with pdfplumber.open(pdf_path) as pdf:
        p = pdf.pages[page_num]
        tables = p.extract_tables()
        parts = []
        for tbl in (tables or []):
            rows = [[str(c or '').strip() for c in row] for row in tbl if row]
            if not rows: continue
            h = rows[0]; sep = ['---']*len(h)
            md = '| ' + ' | '.join(h) + ' |\n| ' + ' | '.join(sep) + ' |\n'
            for row in rows[1:]:
                while len(row) < len(h): row.append('')
                md += '| ' + ' | '.join(row) + ' |\n'
            parts.append(md)
        txt = p.extract_text() or ''
        if txt.strip(): parts.append(txt.strip())
        return '\n\n'.join(parts)

def process_pdf(pdf_path):
    name = Path(pdf_path).stem
    print(f'\n📄 {Path(pdf_path).name}')
    doc = fitz.open(pdf_path)
    parts = [f'# {name}\n']
    for i, page in enumerate(doc):
        raw_text = page.get_text().strip()
        print(f'  Trang {i+1}/{len(doc)}', end=' ')
        if len(raw_text) >= 80:
            print('(text)', end=' ')
            txt = extract_text_page(pdf_path, i)
            if txt.strip(): parts.append(f'## Trang {i+1}\n\n{txt.strip()}')
        else:
            print('(scan→Vision)', end=' ')
            img = page_to_image(page)
            md = gemini_ocr(img)
            if md.strip(): parts.append(f'## Trang {i+1}\n\n{md.strip()}')
            time.sleep(2)
        print('✅')
    doc.close()
    return '\n\n---\n\n'.join(parts)

print('✅ Hàm xử lý đã sẵn sàng')

In [ ]:
# ── CELL 4: Xử lý tất cả PDF ─────────────────────────────────
pdf_files = list(Path(FULL_PATH).rglob('*.pdf')) + list(Path(FULL_PATH).rglob('*.PDF'))
print(f'🔍 Tìm thấy {len(pdf_files)} file PDF')
if not pdf_files:
    print('⚠️ Không có PDF nào. Kiểm tra lại DRIVE_FOLDER.')
else:
    results = {}
    for f in pdf_files:
        try:
            md = process_pdf(str(f))
            results[f.stem + '.md'] = md
            out = os.path.join(OUTPUT_DIR, f.stem + '.md')
            Path(out).write_text(md, encoding='utf-8')
            print(f'  💾 {f.stem}.md ({len(md):,} ký tự)')
        except Exception as e:
            print(f'  ❌ Lỗi: {e}')
    print(f'\n✅ Xong {len(results)}/{len(pdf_files)} files')

In [ ]:
# ── CELL 5: Push lên GitHub ───────────────────────────────────
def gh_push(repo_path, content, msg):
    url = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{repo_path}'
    hdrs = {'Authorization': f'Bearer {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json'}
    sha = None
    r = requests.get(f'{url}?ref={GITHUB_BRANCH}', headers=hdrs)
    if r.ok: sha = r.json().get('sha')
    body = {'message': msg, 'branch': GITHUB_BRANCH,
            'content': base64.b64encode(content.encode()).decode()}
    if sha: body['sha'] = sha
    r = requests.put(url, headers=hdrs, json=body)
    print(f'  {"✅" if r.ok else "❌"} {repo_path}')
    return r.ok

if GITHUB_TOKEN and results:
    print('🚀 Đang push lên GitHub...')
    ok = sum(gh_push(f'data/pdf-extracted/{fn}', content,
                     f'data: add {fn} extracted by Colab Vision')
             for fn, content in results.items())
    time.sleep(1)
    print(f'\n✅ Pushed {ok}/{len(results)} files → data/pdf-extracted/')
    print('⚡ Chạy Chay_Dong_Bo.bat để rebuild RAG index!')
else:
    bk = f'/content/drive/MyDrive/NhaDat/output_markdown'
    os.makedirs(bk, exist_ok=True)
    for fn, content in results.items():
        Path(os.path.join(bk, fn)).write_text(content, encoding='utf-8')
    print(f'💾 Backup vào Drive: {bk}')
    if not GITHUB_TOKEN:
        print('⚠️ Chưa set GITHUB_TOKEN → file lưu ở Drive, copy vào OneDrive rồi chạy .bat')